# 30 · Two-by-two table against statsmodels

**Question.** When Fieldwork reports a risk ratio of **5.35** for cholera death among the people
of sub-districts supplied by the Southwark and Vauxhall Company, with a 95% interval of **3.36 to
8.53**, does an independent implementation get the same numbers?

This widget was built four days before the South London example is shown to FETP-trained
epidemiologists, in a layout they were trained to read. Until this check every number in it
rested on Fieldwork's own tests.

**The table is derived inside Fieldwork, not transcribed.** The six counts come from relating
Snow's Table XII onto his Table VIII, totalling within the water supply, and subtracting deaths
from population — four widgets, three of them new. Before they existed that table was computed
by hand and inlined into the example project with an empty digest.


In [ ]:
# Only what this notebook imports. See 00-kernel-check for what this kernel really has.
NEEDED = ["statsmodels", "scipy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the environment as installed.")


## Parity

`Table2x2` carries the risk ratio and the odds ratio with the interval forms Fieldwork declares —
Katz log and Wald log. **It does not carry a risk difference**, checked against the installed
version rather than assumed, so that one is compared against `confint_proportions_2indep` at
`method='wald'`, the unpooled Wald. `agresti-caffo` is carried beside it as a third opinion in the
role `score` plays in check 27: never asserted equal, so what the declared choice costs is
visible.

The tolerance is **1e-9**, tighter than check 27's 1e-6. That is deliberate: check 27 found the
95% normal quantile hard-coded as 1.9600 where it is 1.959963984540054, and 1e-9 is what detects
that class of defect rather than tolerating it. The quantile now lives in one place,
`src/epi-estimate.ts`, so the next estimate cannot reintroduce it.


In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_30_two_by_two import run

report = run(Path.cwd() / "fixtures" / "two-by-two-snow-001.json")
print("agrees:", report["agrees"])
for t in report["tables"]:
    print(f"\n{t['note']}")
    print(f"  orientation: {'OK' if t['orientation']['agrees'] else 'DIFF'}")
    for e in t["estimates"]:
        f, s = e["fieldwork"], e["statsmodels"]
        print(f"  {'OK ' if e['agrees'] else 'DIFF'} {e['name']:16s} "
              f"{f['value']:.9f} [{f['lower']:.9f}, {f['upper']:.9f}]  "
              f"vs {s['value']:.9f} [{s['lower']:.9f}, {s['upper']:.9f}]")


## What this check found, before the widget shipped

Fieldwork's own documentation explained the 8.40-against-5.35 gap in the South London project as
**rate versus risk**: a rate ratio of 8.40 per house beside a two-by-two where the risk ratio on
people is 5.35.

That is wrong, and the cell below is what showed it. A risk ratio over the same houses is the
rate ratio's value **bit for bit**. Deaths per house *is* a risk when a house can hold at most one
death, so the two are the same division; only the intervals differ, and only in the second
decimal.

**So the gap is the denominator — a house against a person — and not the estimand.** Which makes
the hazard worse rather than milder. A reader who takes 8.40 for the risk ratio beside the houses
table is *right*, and is thereby taught that the substitution is safe. It is safe only while the
outcome is rare. The thing that moves the number is the subject being counted, and that is the one
thing the familiar layout does not show.


In [ ]:
finding = report["denominatorFinding"]
for note, value in finding["riskRatios"].items():
    print(f"{value!r:22} risk ratio — {note}")
print(f"{finding['rateRatioOnHouses']!r:22} RATE ratio over the same houses")

houses = next(t for t in report["tables"] if "houses" in t["note"])
rr = next(e for e in houses["estimates"] if e["name"] == "risk ratio")
print("\nidentical point estimate:",
      rr["fieldwork"]["value"] == finding["rateRatioOnHouses"])
print("Katz   interval:", f"{rr['fieldwork']['lower']:.4f} to {rr['fieldwork']['upper']:.4f}")
print("Poisson interval: 6.8410 to 10.3189   (from rate_ratio, check 27)")


## The label, and where it was read

Every estimate carries the words the familiar layout prints for the same quantity, so a trained
reader can find the row they were looking for:

| computed | printed as |
| --- | --- |
| risk ratio · Katz log | `Risk Ratio (RR)` |
| odds ratio · Wald log | `Odds Ratio (cross product)` |
| risk difference · Wald unpooled | `Risk Difference (RD%)` |

These were **read from Epi Info AI's classic TABLES results panel**, not from Epi Info. That is a
sibling project whose epi kernel was synthesized from the original, so they are a good deduction
and not a verification — and the field `epiInfoLabelSource` records which.

**This check fails if a label ever claims `epi-info` as its source**, because nobody has read Epi
Info itself, and it fails on a bare label with no source. The purpose is unchanged from when the
criterion required these to be *empty*: a field that can be filled unverifiably is worse than no
field. What it polices is now the provenance rather than the absence.

Note the scale: that panel prints `RD%` as a percentage, so Snow's `0.004093` reads there as
`0.4093%`. Same quantity, different scale — stated on the estimate rather than reconciled away.


In [ ]:
for t in report["tables"]:
    for e in t["estimates"]:
        print(f"{e['name']:16} {str(e['epiInfoLabel']):28} {e['epiInfoLabelSource']}")
    break

print("\nlabel problems:", report["epiInfoLabelLeaks"])
print("criterion     :", report["criterion"]["epiInfoLabel"])


## What this cannot establish

**Agreement with Epi Info.** statsmodels is not Epi Info. This project makes no parity claim and
has no parity suite; see `recognisable-not-identical-an-epi-kernel-of-our-own`.

**That the comparison is fair.** The both-companies stratum — 16 of Snow's 32 sub-districts — is
excluded by name, not controlled for. That is Snow's own confounding, and the reason he went to
the house level rather than settling it at the sub-district.

**That the normal approximations are adequate.** The Lambeth-only group has 18 deaths. An exact
interval would be the better comparison there, and it is not implemented.

**Which denominator is right.** Houses and people give 8.40 and 5.35 over the same outbreak. That
choice is the practitioner's, and this check exists partly to make sure nothing in the
application pretends otherwise.
